# 04 · Pune directory, recommendations and model transfer

ParkSmart 2.0 focuses on Pune. No public Pune occupancy data exists, so this notebook (1) audits the hand-curated, sourced Pune facility directory, (2) shows the transparent recommendation logic, and (3) evaluates whether the Birmingham-trained model *transfers* to a car park it has never seen, which is the evidence needed before any Pune forecast could be shown.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from IPython.display import Image, display
from src import config as C

## 1. The Pune directory and its evidence

In [2]:
from src.data_loader import load_pune, summary_counts, STATUS
d = load_pune()
summary_counts(d)

{'facilities': 19,
 'with_location': 15,
 'reported_operational': 4,
 'announced_or_conflicting': 9,
 'with_published_tariff': 11,
 'with_capacity': 2,
 'sources': 40,
 'destinations': 16}

In [3]:
d.facilities[['name','locality','type_label','status_label','vehicles_label','has_tariff','has_location','best_source_kind']]

,name,locality,type_label,status_label,vehicles_label,has_tariff,has_location,best_source_kind
0,PCMC Bhavan metro station parking,Pimpri,Metro station,Conflicting reports,"Cycle, Two-wheeler, Car",True,True,News report
1,Sant Tukaram Nagar metro station parking,Pimpri,Metro station,Announced – opening not confirmed,"Cycle, Two-wheeler, Car",True,True,News report
2,Phugewadi metro station parking,Dapodi,Metro station,Announced – opening not confirmed,"Cycle, Two-wheeler, Car",True,True,News report
3,Bopodi metro station parking,Bopodi,Metro station,Announced – opening not confirmed,"Cycle, Two-wheeler, Car",True,True,News report
4,Shivajinagar metro station parking,Shivajinagar,Metro station,Announced – opening not confirmed,"Cycle, Two-wheeler, Car",True,True,News report
5,District Court (Civil Court) metro station par...,Shivajinagar,Metro station,Announced – opening not confirmed,"Cycle, Two-wheeler, Car",True,True,News report
6,Swargate metro station parking,Swargate,Metro station,Conflicting reports,"Cycle, Two-wheeler, Car",True,True,News report
7,Ideal Colony metro station parking,Kothrud,Metro station,Announced – opening not confirmed,"Cycle, Two-wheeler, Car",True,False,News report
8,Mangalwar Peth (RTO) metro station parking,Sangamvadi,Metro station,Announced – opening not confirmed,"Cycle, Two-wheeler, Car",True,True,News report
9,Kalyani Nagar metro station two-wheeler parking,Kalyani Nagar,Metro station,Reported operational,Two-wheeler,False,True,Wikipedia


In [4]:
d.facilities['status_label'].value_counts()

status_label
Announced – opening not confirmed        7
Reported operational                     4
Conflicting reports                      2
Directory listing only                   2
Contract period ended – check on site    1
Restricted access                        1
Non-operational                          1
Private venue parking                    1
Name: count, dtype: int64

In [5]:
import pandas as pd
pd.Series({k: v['kind'] for k, v in d.sources.items()}).value_counts()

wikipedia    24
news         12
official      2
directory     2
Name: count, dtype: int64

Every row cites at least one dated source; `null` fields mean the sources are silent. Status labels describe the evidence on the source date. Nine of the 19 facilities are metro-station lots announced in Feb 2024 whose opening was never confirmed, two of them contradicted by later reports. That is why the app treats them as *worth checking* and never as available.

### Coordinate sanity check
Every mapped facility should lie within a few km of a cited destination. Wikipedia locality coordinates for Kothrud, Aundh and Pune Camp failed this kind of check and were not used.

In [6]:
from src.location_services import haversine_km
dests = d.destinations.dropna(subset=['lat'])
rows = []
for _, f in d.facilities[d.facilities.has_location].iterrows():
    dist, name = min((haversine_km(f.lat, f.lon, x.lat, x.lon), x['name']) for _, x in dests.iterrows())
    rows.append({'facility': f['name'], 'nearest destination': name, 'km': round(dist, 2)})
pd.DataFrame(rows)

,facility,nearest destination,km
0,PCMC Bhavan metro station parking,Pimpri (PCMC Bhavan metro),0.00
1,Sant Tukaram Nagar metro station parking,Pimpri (PCMC Bhavan metro),2.11
2,Phugewadi metro station parking,Pimpri (PCMC Bhavan metro),5.17
3,Bopodi metro station parking,Shivajinagar,4.32
4,Shivajinagar metro station parking,Shivajinagar,0.56
5,District Court (Civil Court) metro station par...,Shivajinagar,1.52
6,Swargate metro station parking,Swargate,0.08
7,Mangalwar Peth (RTO) metro station parking,Pune Railway Station,0.97
8,Kalyani Nagar metro station two-wheeler parking,Kalyani Nagar (metro station),0.00
9,Pune Railway Station pay-and-park,Pune Railway Station,0.00


## 2. Recommendations
Score = 0.6 × proximity + 0.4 × evidence; hard filters only for known reasons.

In [7]:
from src.recommendations import recommend
dest = d.destinations.set_index('id').loc['fc-road'].to_dict() | {'id': 'fc-road'}
rec = recommend(d, dest, 'car', hours=3, budget=None, radius_km=5)
rec.ranked[['name','distance_km','proximity','evidence','score','cost','missing']].round(2)

,name,distance_km,proximity,evidence,score,cost,missing
0,Shivajinagar metro station parking,1.46,0.71,0.40,0.58,50.0,"live occupancy, capacity"
1,PMC Haribhau Sane Parking Lot,1.39,0.72,0.35,0.57,NaN,"live occupancy, vehicle types, price, capacity..."
2,District Court (Civil Court) metro station par...,2.01,0.60,0.40,0.52,50.0,"live occupancy, capacity"
3,Hutatma Babu Genu Mandai Parking,2.22,0.56,0.35,0.47,NaN,"live occupancy, vehicle types, price, capacity..."
4,Mangalwar Peth (RTO) metro station parking,2.82,0.44,0.40,0.42,50.0,"live occupancy, capacity"
5,Pune Railway Station pay-and-park,3.74,0.25,0.50,0.35,NaN,"live occupancy, price, capacity, opening hours"
6,Swargate metro station parking,3.35,0.33,0.20,0.28,50.0,"live occupancy, capacity"


In [8]:
rec.not_suitable[['name','distance_km','reason']].round(2).head(10)

,name,distance_km,reason
0,Swargate bus station parking,3.42,Restricted access
1,Bopodi metro station parking,5.09,5.1 km away (beyond 5 km)
2,Kalyani Nagar metro station two-wheeler parking,7.37,No car parking listed; 7.4 km away (beyond 5 km)
3,Phugewadi metro station parking,7.52,7.5 km away (beyond 5 km)
4,Phoenix Marketcity mall parking,9.20,9.2 km away (beyond 5 km)
5,Sant Tukaram Nagar metro station parking,10.38,10.4 km away (beyond 5 km)
6,Pune Airport multi-level car park,10.68,10.7 km away (beyond 5 km)
7,PCMC Bhavan metro station parking,12.32,12.3 km away (beyond 5 km)


Budget and vehicle filters exclude only facilities where the *known* price or vehicle list rules them out:

In [9]:
r = recommend(d, d.destinations.set_index('id').loc['shivajinagar'].to_dict() | {'id': 'shivajinagar'}, 'car', 3, budget=40)
r.not_suitable[['name','reason']].head(6)

,name,reason
0,Shivajinagar metro station parking,₹50 is over your ₹40 budget
1,District Court (Civil Court) metro station par...,₹50 is over your ₹40 budget
2,Mangalwar Peth (RTO) metro station parking,₹50 is over your ₹40 budget
3,Swargate metro station parking,₹50 is over your ₹40 budget
4,Swargate bus station parking,Restricted access
5,Bopodi metro station parking,₹50 is over your ₹40 budget


## 3. Uncertainty by time of day
Split-conformal ranges from walk-forward residuals, one width per band of the forecast time.

In [10]:
ev = json.loads((C.METRICS_DIR / 'evaluation_summary.json').read_text())
pd.DataFrame(ev['60']['interval_band_report']['by_band']).T.round(3)

,n,global_coverage,band_coverage,band_halfwidth_pp
08:30–10:30,1409.0,0.881,0.909,5.190
11:00–12:30,1406.0,0.864,0.876,4.721
13:00–14:30,1433.0,0.930,0.900,3.994
15:00–16:30,1491.0,0.926,0.905,4.130


A single width under-covers the morning fill-up and over-covers the afternoon; band widths bring every band closer to the 90% target while keeping overall coverage near 90%.

## 4. Does the model transfer to an unseen car park?
A park-agnostic model (ID hidden) is trained on 27 car parks and tested on the 28th, exactly as an uploaded Pune file would be processed.

In [11]:
t = json.loads((C.METRICS_DIR / 'transfer_summary.json').read_text())
pd.DataFrame({h: t[h] for h in ('30', '60')})

,30,60
model,HistGradientBoosting (park-agnostic),HistGradientBoosting (park-agnostic)
holdout_park,Others-CCCPS8,Others-CCCPS8
holdout_rows,1221,1150
holdout_MAE_pp,1.103976,1.746484
holdout_persistence_MAE_pp,2.68579,5.283628
holdout_interval_coverage,0.969697,0.976522
park_cv_interval_global_pp,3.308432,5.630327


In [12]:
from src.transfer import evaluate_upload, SAMPLE_CSV
res = evaluate_upload(pd.read_csv(SAMPLE_CSV), 60)
{'model_MAE_pp': res.model_mae_pp, 'persistence_MAE_pp': res.persistence_mae_pp, 'coverage': res.coverage, 'beats_persistence': res.beats_persistence, 'rows_scored': res.n_scored}

{'model_MAE_pp': 1.7464840196700662,
 'persistence_MAE_pp': 5.283628231270144,
 'coverage': 0.9765217391304348,
 'beats_persistence': True,
 'rows_scored': 1150}

**Interpretation.** On a car park it never saw, the park-agnostic model is about three times more accurate than persistence. This supports *testing* it on Pune counts, and nothing more: the held-out car park shares the training period and city, which makes it easier than Pune in a different year. The upload tool therefore measures error on the user's own data and only shows forecasts if the model beats persistence there.